In [22]:
# Script to automate downloading of Global FWI data from Zenodo repositories
# See https://zenodo.org/record/3626193
# Author: Piyush Jain
# Date: 24/01/2020

# Usage
# python fwi_download.py

import requests
import xarray as xr
import pandas as pd

In [2]:
# set up dictionary mapping variable to download paths
dict_paths = {}

#FFMC using default DC start-up
dict_paths["FFMC_dDC"] = "https://zenodo.org/record/3540950/files/no_overwintering_fine_fuel_moisture_code_1979.nc?download=1"

#DMC using default DC start-up
dict_paths["DMC_dDC"] = "https://zenodo.org/record/3540954/files/no_overwintering_duff_moisture_code_1979.nc?download=1"

#DC using default DC start-up
dict_paths["DC_dDC"] = "https://zenodo.org/record/3540959/files/no_overwintering_drought_code_1979.nc?download=1"

#ISI using default DC start-up
dict_paths["ISI_dDC"] = "https://zenodo.org/record/3540946/files/no_overwintering_initial_spread_index_1979.nc?download=1"

#BUI using default DC start-up
dict_paths["BUI_dDC"] = "https://zenodo.org/record/3540942/files/no_overwintering_build_up_index_1979.nc?download=1"

#FWI using default DC start-up
dict_paths["FWI_dDC"] = "https://zenodo.org/record/3540938/files/no_overwintering_fire_weather_index_1979.nc?download=1"

#DSR using default DC start-up
dict_paths["DSR_dDC"] = "https://zenodo.org/record/3540962/files/no_overwintering_daily_severity_rating_1979.nc?download=1"

#FFMC using overwintered DC start-up
dict_paths["FFMC_owDC"] = "https://zenodo.org/record/3540922/files/fine_fuel_moisture_code_1979.nc?download=1"

#DMC using overwintered DC start-up
dict_paths["DMC_owDC"] = "https://zenodo.org/record/3540924/files/duff_moisture_code_1979.nc?download=1"

#DC using overwintered DC start-up
dict_paths["DC_owDC"] = "https://zenodo.org/record/3540926/files/drought_code_1979.nc?download=1"

#ISI using overwintered DC start-up
dict_paths["ISI_owDC"] = "https://zenodo.org/record/3540920/files/initial_spread_index_1979.nc?download=1"

#BUI using overwintered DC start-up
dict_paths["BUI_owDC"] = "https://zenodo.org/record/3540918/files/build_up_index_1979.nc?download=1"

#FWI using overwintered DC start-up
dict_paths["FWI_owDC"] = "https://zenodo.org/record/3539654/files/fire_weather_index_1979.nc?download=1"

#DSR using overwintered DC start-up
dict_paths["DSR_owDC"] = "https://zenodo.org/record/3540928/files/daily_severity_rating_1979.nc?download=1"


In [3]:
# edit parameters:
year_start = 1979
year_end = 2018
overwinter_DC = True
variable = "DC"

# which version of data do we want
suffix = "_owDC"
if not overwinter_DC:
    suffix = "_dDC"


In [7]:
for year in range(1997, year_end+1):

    # data url
    url = dict_paths[variable+suffix]
    url = url.replace("1979", str(year))
    # save filename
    fname =  url.split("/")[-1]
    fname = fname.split("?")[0]                                                                                                   
    print("Downloading " + fname)
    # Download the file
    r = requests.get(url)
    with open("/nesi/project/niwa00015/queenle/data/fire/"+variable+"/" + fname, 'wb') as f:
        f.write(r.content)
        

In [34]:
'''
Read yearly netcdfs and CREATE 1980-2020 netcdf with full dates
'''

var = 'FFMC'
var_full = 'fine_fuel_moisture_code_'
ds_list = []
for year in range(1980,2019):
    ds = xr.open_dataset("/nesi/project/niwa00015/queenle/data/fire/" + var + "/" + var_full + str(year) + ".nc")
    ds = ds.sel(Latitude=slice(55,30), Longitude=slice(225,265))
    ds['Time'] = [pd.to_datetime(day-1, unit='D', origin=str(year)) for day in ds.Time.values.tolist()]
    
    ds_list.append(ds)
    
var_1980_2018 = xr.concat(ds_list,dim='Time')

var_1980_2018.to_netcdf("/nesi/project/niwa00015/queenle/data/fire/" + var + "/" + var + "_1980_2018.nc")
